In [1]:
# =========================================================
# 01_data_audit.ipynb
# Purpose:
#   - Load the 3 raw datasets
#   - Inspect their structure
#   - Identify text columns and label columns
#   - Check missing values
#   - Check duplicates
#   - Check label values and class distributions
#   - Prepare for the cleaning/merge notebook
# =========================================================

In [2]:
# Standard library
from pathlib import Path
import json

# Data analysis
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt

# Optional display improvements
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
# ---------------------------------------------------------
# Define the base paths of the ML workspace
# ---------------------------------------------------------

# This assumes your notebook is inside: ml/notebooks/
# So ml/ is one folder above
ML_ROOT = Path.cwd().parent

RAW_DATA_DIR = ML_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = ML_ROOT / "data" / "processed"
FINAL_DATA_DIR = ML_ROOT / "data" / "final"

print("ML root:", ML_ROOT)
print("Raw data directory:", RAW_DATA_DIR)
print("Processed data directory:", PROCESSED_DATA_DIR)
print("Final data directory:", FINAL_DATA_DIR)

ML root: C:\Users\hp\Lucidia-project\ml
Raw data directory: C:\Users\hp\Lucidia-project\ml\data\raw
Processed data directory: C:\Users\hp\Lucidia-project\ml\data\processed
Final data directory: C:\Users\hp\Lucidia-project\ml\data\final


In [4]:
# ---------------------------------------------------------
# List all files in the raw data folder
# ---------------------------------------------------------

raw_files = list(RAW_DATA_DIR.glob("*"))

print("Files found in raw data folder:")
for file_path in raw_files:
    print("-", file_path.name)

Files found in raw data folder:
- Algerian Review.csv
- data_2.csv
- data_3.csv


In [5]:
# ---------------------------------------------------------
# Define exact dataset file paths
# Update these names only if your filenames differ
# ---------------------------------------------------------

file_1 = RAW_DATA_DIR / "Algerian Review.csv"
file_2 = RAW_DATA_DIR / "data_2.csv"
file_3 = RAW_DATA_DIR / "data_3.csv"

print(file_1.exists(), "-", file_1.name)
print(file_2.exists(), "-", file_2.name)
print(file_3.exists(), "-", file_3.name)

True - Algerian Review.csv
True - data_2.csv
True - data_3.csv


In [6]:
# ---------------------------------------------------------
# Load the CSV files into pandas DataFrames
# We use encoding fallback logic because CSV files can vary
# ---------------------------------------------------------

def load_csv_safely(file_path: Path) -> pd.DataFrame:
    """
    Try loading a CSV file using common encodings.
    Raises the last exception if all attempts fail.
    """
    encodings_to_try = ["utf-8", "utf-8-sig", "latin1", "cp1252"]

    last_error = None
    for enc in encodings_to_try:
        try:
            df = pd.read_csv(file_path, encoding=enc)
            print(f"Loaded {file_path.name} with encoding: {enc}")
            return df
        except Exception as e:
            last_error = e

    raise last_error

df1 = load_csv_safely(file_1)
df2 = load_csv_safely(file_2)
df3 = load_csv_safely(file_3)

print("\nShapes:")
print("df1:", df1.shape)
print("df2:", df2.shape)
print("df3:", df3.shape)

Loaded Algerian Review.csv with encoding: utf-8
Loaded data_2.csv with encoding: utf-8
Loaded data_3.csv with encoding: utf-8

Shapes:
df1: (5320, 6)
df2: (1262, 5)
df3: (5782, 7)


In [7]:
# ---------------------------------------------------------
# Display the first few rows of each dataset
# ---------------------------------------------------------

print("========== DATASET 1 ==========")
display(df1.head())

print("========== DATASET 2 ==========")
display(df2.head())

print("========== DATASET 3 ==========")
display(df3.head())

========== DATASET 1 ==========


,name,date,likes,comment,sentiment,source
0,djezzy.dz,04/03/20 10:29:29,2309,استفيدوا من عروض جازي عايلة الجديدة لي توالم كامل افراد العائلة 👪 راهي متوفرة في كامل وكالات جازي مع شريحة SIM مجّانية 😱👌🏻 Bénéficiez des nouveaux forfaits DJEZZY 3AYLA qui conviennent à tous les...,0,insta
1,amira_la_bonoise,04/03/20 10:31:18,0,Conx ta3koum dayra ki lhaaaam w say,-1,insta
2,noureddiiiiiine,04/03/20 10:31:38,0,@zaki_medjber_62,0,insta
3,linaragguem,04/03/20 10:32:28,0,ما فهمتش هي sim وحدة يخدمو بها العائلة كاملة,0,insta
4,samo_oussama,04/03/20 10:36:29,0,🙏😉🙏,1,insta


========== DATASET 2 ==========


,ID,comment,updated_at,like_count,sentiment
0,0,نفس شي مع la win\nتقولو فيسبوك مجاني ....بصح غير يخلاصو لجيغا متقدرش تتفرج فيديوهات,2022-03-30T16:54:00Z,3,1
1,1,غير نلقاو الكونيكسيو و من بعد ساهل,2022-03-30T16:23:00Z,2,1
2,2,نقدر نحول مبتسم تاعي ل سما,2022-03-30T17:08:00Z,0,0
3,3,نسقسيكم برك كان عندي 16 نيوم ملي فليكسيت 1000 دج و اكتيفيت لاوين 650 دج درك لقيت 1 دج 350دج وين راحو ماني دايرة لا رنتي لا هم يحزنون وين راحو دراهمي,2022-03-30T16:54:00Z,2,1
4,4,علاه تخلو الناس تفليكسي 🙄من بعد تدولهم كيما انا منيش دايرة رناتي و كلما نفليكسي تدولي 🙄 ؟!!,2022-03-31T14:15:00Z,2,1


========== DATASET 3 ==========


,Unnamed: 0.2,Unnamed: 0,Unnamed: 0.1,comment,updated_at,like_count,sentiment
0,0,0,0.0,بالنسبة لجماعة هايلة بزاف عرض 1200الف اكتيفيه ب امتياز #444*يعطوك 30جيقا ماشي 20جيقا وشكرا خاوتي.........,2020-07-25T13:06:49Z,24.0,0.0
1,1,1,1.0,نحوها,2022-03-07T17:37:38Z,0.0,0.0
2,2,2,2.0,ماكسي بيسك بزاف مافيهاش 20g,2021-08-18T18:46:32Z,0.0,1.0
3,3,3,3.0,انا عندي هايلة و الماكسي و في 2 في حالة..الحل الوحيد هو لازم يدخلو الشركات الاجنبية كيما اورنج طيح عليهم البق سرعة انترنت خرافية و سعر مغري .,2019-11-19T20:01:56Z,11.0,0.0
4,4,4,4.0,ههههه جيت هنا,2021-10-19T01:45:12Z,0.0,0.0


In [9]:
# ---------------------------------------------------------
# Print the column names of each dataset
# This is one of the most important audit steps
# ---------------------------------------------------------

print("Columns in dataset 1:")
print(df1.columns.tolist())

print("\nColumns in dataset 2:")
print(df2.columns.tolist())

print("\nColumns in dataset 3:")
print(df3.columns.tolist())

Columns in dataset 1:
['name', 'date', 'likes', 'comment', 'sentiment', 'source']

Columns in dataset 2:
['ID', 'comment', 'updated_at', 'like_count', 'sentiment']

Columns in dataset 3:
['Unnamed: 0.2', 'Unnamed: 0', 'Unnamed: 0.1', 'comment', 'updated_at', 'like_count', 'sentiment']


In [10]:
# ---------------------------------------------------------
# A helper function to summarize a dataframe quickly
# ---------------------------------------------------------

def summarize_dataframe(df: pd.DataFrame, name: str) -> pd.DataFrame:
    summary = pd.DataFrame({
        "column": df.columns,
        "dtype": [str(df[col].dtype) for col in df.columns],
        "missing_count": [df[col].isna().sum() for col in df.columns],
        "missing_percent": [round(df[col].isna().mean() * 100, 2) for col in df.columns],
        "n_unique": [df[col].nunique(dropna=True) for col in df.columns],
    })
    summary.insert(0, "dataset", name)
    return summary

summary_df1 = summarize_dataframe(df1, "dataset_1")
summary_df2 = summarize_dataframe(df2, "dataset_2")
summary_df3 = summarize_dataframe(df3, "dataset_3")

audit_summary = pd.concat([summary_df1, summary_df2, summary_df3], ignore_index=True)

display(audit_summary)

,dataset,column,dtype,missing_count,missing_percent,n_unique
0,dataset_1,name,str,0,0.00,3348
1,dataset_1,date,str,54,1.02,5000
2,dataset_1,likes,int64,0,0.00,77
3,dataset_1,comment,str,0,0.00,4747
4,dataset_1,sentiment,int64,0,0.00,3
5,dataset_1,source,str,0,0.00,2
6,dataset_2,ID,int64,0,0.00,1260
7,dataset_2,comment,str,0,0.00,1233
8,dataset_2,updated_at,str,0,0.00,1167
9,dataset_2,like_count,int64,0,0.00,26


In [11]:
# ---------------------------------------------------------
# A helper function to summarize a dataframe quickly
# ---------------------------------------------------------

def summarize_dataframe(df: pd.DataFrame, name: str) -> pd.DataFrame:
    summary = pd.DataFrame({
        "column": df.columns,
        "dtype": [str(df[col].dtype) for col in df.columns],
        "missing_count": [df[col].isna().sum() for col in df.columns],
        "missing_percent": [round(df[col].isna().mean() * 100, 2) for col in df.columns],
        "n_unique": [df[col].nunique(dropna=True) for col in df.columns],
    })
    summary.insert(0, "dataset", name)
    return summary

summary_df1 = summarize_dataframe(df1, "dataset_1")
summary_df2 = summarize_dataframe(df2, "dataset_2")
summary_df3 = summarize_dataframe(df3, "dataset_3")

audit_summary = pd.concat([summary_df1, summary_df2, summary_df3], ignore_index=True)

display(audit_summary)

,dataset,column,dtype,missing_count,missing_percent,n_unique
0,dataset_1,name,str,0,0.00,3348
1,dataset_1,date,str,54,1.02,5000
2,dataset_1,likes,int64,0,0.00,77
3,dataset_1,comment,str,0,0.00,4747
4,dataset_1,sentiment,int64,0,0.00,3
5,dataset_1,source,str,0,0.00,2
6,dataset_2,ID,int64,0,0.00,1260
7,dataset_2,comment,str,0,0.00,1233
8,dataset_2,updated_at,str,0,0.00,1167
9,dataset_2,like_count,int64,0,0.00,26


In [12]:
# ---------------------------------------------------------
# More detailed schema information
# ---------------------------------------------------------

print("========== INFO: DATASET 1 ==========")
df1.info()

print("\n========== INFO: DATASET 2 ==========")
df2.info()

print("\n========== INFO: DATASET 3 ==========")
df3.info()

========== INFO: DATASET 1 ==========
<class 'pandas.DataFrame'>
RangeIndex: 5320 entries, 0 to 5319
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   name       5320 non-null   str  
 1   date       5266 non-null   str  
 2   likes      5320 non-null   int64
 3   comment    5320 non-null   str  
 4   sentiment  5320 non-null   int64
 5   source     5320 non-null   str  
dtypes: int64(2), str(4)
memory usage: 894.2 KB

========== INFO: DATASET 2 ==========
<class 'pandas.DataFrame'>
RangeIndex: 1262 entries, 0 to 1261
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   ID          1262 non-null   int64
 1   comment     1262 non-null   str  
 2   updated_at  1262 non-null   str  
 3   like_count  1262 non-null   int64
 4   sentiment   1262 non-null   int64
dtypes: int64(3), str(2)
memory usage: 182.2 KB

========== INFO: DATASET 3 ==========
<class 'pandas.DataFr

In [13]:
# ---------------------------------------------------------
# Check for columns like "Unnamed: 0"
# These often come from saved pandas indexes
# ---------------------------------------------------------

def find_unnamed_columns(df: pd.DataFrame):
    return [col for col in df.columns if str(col).lower().startswith("unnamed")]

print("Unnamed columns in dataset 1:", find_unnamed_columns(df1))
print("Unnamed columns in dataset 2:", find_unnamed_columns(df2))
print("Unnamed columns in dataset 3:", find_unnamed_columns(df3))

Unnamed columns in dataset 1: []
Unnamed columns in dataset 2: []
Unnamed columns in dataset 3: ['Unnamed: 0.2', 'Unnamed: 0', 'Unnamed: 0.1']


In [15]:
# ---------------------------------------------------------
# Try to identify likely label columns by searching
# common names such as: label, sentiment, annotation, class
# ---------------------------------------------------------

candidate_keywords = ["label", "sentiment", "annotation", "class", "polarity", "target"]

def find_candidate_label_columns(df: pd.DataFrame):
    candidates = []
    for col in df.columns:
        col_lower = str(col).lower()
        if any(keyword in col_lower for keyword in candidate_keywords):
            candidates.append(col)
    return candidates

print("Candidate label columns in dataset 1:", find_candidate_label_columns(df1))
print("Candidate label columns in dataset 2:", find_candidate_label_columns(df2))
print("Candidate label columns in dataset 3:", find_candidate_label_columns(df3))

Candidate label columns in dataset 1: ['sentiment']
Candidate label columns in dataset 2: ['sentiment']
Candidate label columns in dataset 3: ['sentiment']


In [16]:
# ---------------------------------------------------------
# Try to identify likely text/comment columns
# ---------------------------------------------------------

text_keywords = ["text", "comment", "review", "sentence", "content", "message"]

def find_candidate_text_columns(df: pd.DataFrame):
    candidates = []
    for col in df.columns:
        col_lower = str(col).lower()
        if any(keyword in col_lower for keyword in text_keywords):
            candidates.append(col)
    return candidates

print("Candidate text columns in dataset 1:", find_candidate_text_columns(df1))
print("Candidate text columns in dataset 2:", find_candidate_text_columns(df2))
print("Candidate text columns in dataset 3:", find_candidate_text_columns(df3))

Candidate text columns in dataset 1: ['comment']
Candidate text columns in dataset 2: ['comment']
Candidate text columns in dataset 3: ['comment']


In [17]:
# ---------------------------------------------------------
# Show unique values for columns with small numbers of distinct values
# This often reveals the label column
# ---------------------------------------------------------

def inspect_small_cardinality_columns(df: pd.DataFrame, name: str, max_unique: int = 20):
    print(f"\n===== Small-cardinality columns in {name} =====")
    for col in df.columns:
        unique_values = df[col].dropna().unique()
        if len(unique_values) <= max_unique:
            print(f"\nColumn: {col}")
            print("Unique values:", unique_values)

inspect_small_cardinality_columns(df1, "dataset_1")
inspect_small_cardinality_columns(df2, "dataset_2")
inspect_small_cardinality_columns(df3, "dataset_3")


===== Small-cardinality columns in dataset_1 =====

Column: sentiment
Unique values: [ 0 -1  1]

Column: source
Unique values: <ArrowStringArray>
['insta', 'fb']
Length: 2, dtype: str

===== Small-cardinality columns in dataset_2 =====

Column: sentiment
Unique values: [ 1  0  2 11]

===== Small-cardinality columns in dataset_3 =====

Column: sentiment
Unique values: [0. 1. 2.]


In [18]:
# ---------------------------------------------------------
# Show unique values for columns with small numbers of distinct values
# This often reveals the label column
# ---------------------------------------------------------

def inspect_small_cardinality_columns(df: pd.DataFrame, name: str, max_unique: int = 20):
    print(f"\n===== Small-cardinality columns in {name} =====")
    for col in df.columns:
        unique_values = df[col].dropna().unique()
        if len(unique_values) <= max_unique:
            print(f"\nColumn: {col}")
            print("Unique values:", unique_values)

inspect_small_cardinality_columns(df1, "dataset_1")
inspect_small_cardinality_columns(df2, "dataset_2")
inspect_small_cardinality_columns(df3, "dataset_3")


===== Small-cardinality columns in dataset_1 =====

Column: sentiment
Unique values: [ 0 -1  1]

Column: source
Unique values: <ArrowStringArray>
['insta', 'fb']
Length: 2, dtype: str

===== Small-cardinality columns in dataset_2 =====

Column: sentiment
Unique values: [ 1  0  2 11]

===== Small-cardinality columns in dataset_3 =====

Column: sentiment
Unique values: [0. 1. 2.]
